In [1]:
# =====================================================================
# Part II - Image Colorization - TEMPLATE
# =====================================================================
#
# Task: take a grayscale image and predict its colors.
#
# You build and train the model any way you want (autoencoder, VAE, GAN, ...).
#
# The grader will:
#   1. run your model class + load_model() to load your saved weights,
#   2. call colorize() on their own images,
#   3. compare your output to hidden color images (PSNR / MSE).
#
# So you MUST keep the 3 fixed rules below.
# =====================================================================
#
# ---------------------------------------------------------------------
# FIXED RULES (do not change)
# ---------------------------------------------------------------------
#
# 1. Save your trained weights as a state_dict:
#        torch.save(model.state_dict(), "weights.pth")
#    Submit this "weights.pth" file together with your notebook.
#
# 2. All images are 256 x 256 PNG.
#    colorize() input  : grayscale array, shape (256, 256), float in [0, 1]
#    colorize() output : RGB array,       shape (256, 256, 3), float in [0, 1]
#
# 3. Do file loading OUTSIDE colorize() (see the demo at the bottom).
#    colorize() only takes arrays, not file paths.
# ---------------------------------------------------------------------

In [2]:
import numpy as np
import torch
import torch.nn as nn

IMG_SIZE = 256

In [3]:
# ---------------------------------------------------------------------
# 1) YOUR MODEL
# ---------------------------------------------------------------------
# A U-Net that predicts only CHROMINANCE (Cb, Cr), not full RGB.
#
# Plain RGB regression with L1/MSE loss is known to produce muted,
# undersaturated colors: the model hedges toward "safe" averaged
# colors whenever it's unsure, because errors in R/G/B are entangled
# with brightness errors too. The standard fix in the colorization
# literature is to work in a luma/chroma colorspace (Y/Cb/Cr): the
# grayscale input IS (almost exactly) the Y channel already, so it
# doesn't need to be predicted at all -- only the 2 color channels
# (Cb, Cr) do. This confines all prediction error to color, never
# brightness, which is both an easier learning problem and closer to
# how the eye actually perceives color images.
#
# base=32 (from 24, increased for better capacity): with Part I finished, the full machine is
# available, so this affords a bigger network than the base=16 first
# attempt. The default here MUST match what's actually trained --
# load_model() below rebuilds with no arguments, so a mismatched
# default would fail to load the saved weights.

Y_R, Y_G, Y_B = 0.299, 0.587, 0.114  # matches PIL's L = ITU-R 601-2 luma


def rgb_to_ycbcr(rgb):
    """rgb: (..., H, W, 3) in [0,1] -> y, cb, cr each (..., H, W) in [0,1]."""
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    y = Y_R * r + Y_G * g + Y_B * b
    cb = -0.168736 * r - 0.331264 * g + 0.5 * b + 0.5
    cr = 0.5 * r - 0.418688 * g - 0.081312 * b + 0.5
    return y, cb, cr


def ycbcr_to_rgb(y, cb, cr):
    """y, cb, cr: (..., H, W) in [0,1] -> rgb (..., H, W, 3) in [0,1]."""
    cb0, cr0 = cb - 0.5, cr - 0.5
    r = y + 1.402 * cr0
    g = y - 0.344136 * cb0 - 0.714136 * cr0
    b = y + 1.772 * cb0
    return np.clip(np.stack([r, g, b], axis=-1), 0.0, 1.0)


def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
    )


class ColorizeModel(nn.Module):
    def __init__(self, base=32): # Increased base from 24 to 32
        super().__init__()
        self.enc1 = conv_block(1, base)
        self.enc2 = conv_block(base, base * 2)
        self.enc3 = conv_block(base * 2, base * 4)
        self.enc4 = conv_block(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = conv_block(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = conv_block(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = conv_block(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = conv_block(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = conv_block(base * 2, base)

        self.out = nn.Conv2d(base, 2, 1)  # Cb, Cr only

    def forward(self, x):
        # x: (batch, 1, 256, 256) -- the Y (luminance) channel
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out(d1))  # (batch, 2, 256, 256): cb, cr in [0,1]


In [4]:
import torchvision.models as models
import torchvision.transforms as T # Added missing import

class PerceptualLoss(nn.Module):
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        vgg = models.vgg19(pretrained=True).features
        # We only need the features up to a certain layer. Here, we'll use relu4_4
        self.vgg_features = nn.Sequential(*list(vgg.children())[:35]).eval()
        for param in self.vgg_features.parameters():
            param.requires_grad = False

        # Normalize RGB images for VGG input
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self.l1_loss = nn.L1Loss()

    def forward(self, y_gray, pred_cbcr, gt_cbcr):
        # y_gray: (batch, 1, H, W)
        # pred_cbcr, gt_cbcr: (batch, 2, H, W)

        # Convert predicted and ground truth CbCr to RGB using the original Y (luminance)
        # Need to move to CPU for numpy conversion and then back to GPU for VGG
        y_np = y_gray.squeeze(1).cpu().numpy() # (batch, H, W)

        # Detach tensors before converting to numpy
        pred_cb = pred_cbcr[:, 0, :, :].cpu().detach().numpy()
        pred_cr = pred_cbcr[:, 1, :, :].cpu().detach().numpy()
        gt_cb = gt_cbcr[:, 0, :, :].cpu().detach().numpy()
        gt_cr = gt_cbcr[:, 1, :, :].cpu().detach().numpy()

        # Loop through batch to perform ycbcr_to_rgb conversion
        pred_rgb_list = []
        gt_rgb_list = []
        for i in range(y_np.shape[0]):
            pred_rgb_list.append(torch.from_numpy(ycbcr_to_rgb(y_np[i], pred_cb[i], pred_cr[i])))
            gt_rgb_list.append(torch.from_numpy(ycbcr_to_rgb(y_np[i], gt_cb[i], gt_cr[i])))

        pred_rgb = torch.stack(pred_rgb_list).permute(0, 3, 1, 2).to(y_gray.device) # (batch, 3, H, W)
        gt_rgb = torch.stack(gt_rgb_list).permute(0, 3, 1, 2).to(y_gray.device) # (batch, 3, H, W)

        # Normalize RGB images for VGG input
        pred_rgb = self.normalize(pred_rgb)
        gt_rgb = self.normalize(gt_rgb)

        # Extract features
        pred_features = self.vgg_features(pred_rgb)
        gt_features = self.vgg_features(gt_rgb)

        # Calculate L1 loss on features
        perceptual_loss = self.l1_loss(pred_features, gt_features)

        # Combine with original L1 loss on CbCr for local accuracy
        # We can also add the original L1 loss on CbCr to ensure color accuracy locally.
        # This part assumes we want to add to existing L1_loss defined in training.
        # For now, let's return only perceptual loss to observe its effect.
        return perceptual_loss


In [5]:
# ---------------------------------------------------------------------
# 2) LOAD YOUR TRAINED WEIGHTS
# ---------------------------------------------------------------------
# Rebuilds the empty model and loads the saved numbers.
# This is instant - no training.
def load_model(weights_path="weights.pth"):
    model = ColorizeModel()
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    return model

In [6]:
# ---------------------------------------------------------------------
# 3) COLORIZE ONE IMAGE
# ---------------------------------------------------------------------
# gray_img: numpy array, shape (256, 256), float in [0, 1]
# returns : numpy array, shape (256, 256, 3), float in [0, 1]
#
# The external contract is unchanged (grayscale in, RGB out) -- the
# Y/Cb/Cr split is purely an internal implementation detail. gray_img
# is used directly as Y (it already IS the luminance channel), the
# model predicts Cb/Cr, and the three are recombined into RGB.
def colorize(gray_img, model):
    x = torch.from_numpy(gray_img).float().view(1, 1, IMG_SIZE, IMG_SIZE)

    with torch.no_grad():           # no gradients needed for inference
        cb_cr = model(x)            # (1, 2, 256, 256)

    cb = cb_cr[0, 0].cpu().numpy()
    cr = cb_cr[0, 1].cpu().numpy()
    return ycbcr_to_rgb(gray_img, cb, cr)

In [7]:
# =======================================================================
# TRAINING -- produces weights.pth
# =======================================================================
# Everything above this line is the fixed submission interface. Below
# is how weights.pth was actually produced: dataset, training loop,
# and a quick self-check against the same PSNR/MSE metric the grader
# uses, before the final demo.
#
# No training images were provided for this assignment ("you may
# choose all the images you want to train your model" -- directives.txt).
# Flowers102 (torchvision, auto-downloads) was used: colorful, diverse
# natural photos, no license friction, no manual collection needed.
# Only the images are used -- the 102 flower classes are irrelevant to
# colorization and are never touched.

import os
import glob
import random
import torchvision
import torchvision.transforms as T
from PIL import Image

DATA_ROOT = "colorization_data"
os.makedirs(DATA_ROOT, exist_ok=True)
for split in ["train", "val", "test"]:
    torchvision.datasets.Flowers102(root=DATA_ROOT, split=split, download=True)
IMAGE_DIR = os.path.join(DATA_ROOT, "flowers-102", "jpg")

image_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))
print("images found", len(image_paths))

random.Random(0).shuffle(image_paths)
# Use all available images (8189) for training to maximize generalization and quality
N_IMAGES = len(image_paths) # Changed to use all available images
image_paths = image_paths[:N_IMAGES]
n_val = max(1, int(0.1 * len(image_paths)))
val_paths, train_paths = image_paths[:n_val], image_paths[n_val:]
print("train:", len(train_paths), " val:", len(val_paths))


class ColorizationDataset(torch.utils.data.Dataset):
    """Loads a color photo, returns (Y, [Cb, Cr]), all 256x256 in [0, 1].

    Y is the grayscale input (== luminance); Cb/Cr are the color
    channels the model must predict. The photo supervises itself --
    no separate labels needed. `augment=True` applies random
    transformations (train split only) for free extra variety from
    the same images.
    """

    def __init__(self, paths, size=IMG_SIZE, augment=False):
        self.paths = paths
        self.size = size

        if augment:
            self.transform = T.Compose([
                T.RandomResizedCrop(size, scale=(0.8, 1.0), ratio=(0.75, 1.33)), # Random crop and resize
                T.RandomRotation(degrees=15), # Random rotation by up to 15 degrees
                T.RandomHorizontalFlip(p=0.5), # Random horizontal flip
                T.ToTensor() # Convert PIL Image to PyTorch Tensor
            ])
        else:
            self.transform = T.Compose([
                T.Resize((size, size)),
                T.ToTensor()
            ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img_tensor = self.transform(img) # Apply transforms here

        # Convert back to numpy to use existing ycbcr conversion functions
        rgb = img_tensor.permute(1, 2, 0).numpy() # (H, W, C) float in [0, 1]

        y, cb, cr = rgb_to_ycbcr(rgb)
        y_t = torch.from_numpy(y).float().unsqueeze(0)
        cbcr_t = torch.from_numpy(np.stack([cb, cr], axis=0)).float()
        return y_t, cbcr_t

train_ds = ColorizationDataset(train_paths, augment=True)
val_ds = ColorizationDataset(val_paths, augment=False)


images found 8189
train: 7371  val: 818


In [ ]:
BATCH_SIZE = 8
EPOCHS = 100 # Increased epochs from 40 to 100
LR = 1e-3

torch.set_num_threads(6)
torch.manual_seed(0)
model = ColorizeModel()
opt = torch.optim.Adam(model.parameters(), lr=LR)
# Cosine LR decay -- the first attempt trained at a constant LR=1e-3 the
# whole run and the loss barely moved after epoch 0 (measured: epoch 0
# val L1 0.0601, epoch 3 val L1 0.0597 -- essentially flat). A constant
# LR that's fine for the initial big steps is too coarse to keep
# refining once the model is in the right neighborhood; decaying it
# lets training keep making real progress instead of stalling early.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
l1_loss_fn = nn.L1Loss()  # L1 over MSE: sharper edges, less blurring
perceptual_loss_fn = PerceptualLoss() # Instantiate perceptual loss

# Weights for combining losses. Tuned these for better perceptual quality.
L1_WEIGHT = 0.5 # Decreased L1 weight
PERCEPTUAL_WEIGHT = 1.0 # Increased Perceptual weight significantly

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
perceptual_loss_fn.to(device) # Move perceptual loss to device as well
print("device:", device)

# Save the checkpoint with the BEST validation loss, not just whichever
# epoch happens to run last -- with 40 epochs on a modest dataset the
# model can start overfitting the training photos before the end, and
# the grader colorizes *their own* images, not ours. This is the
# generalization safeguard: what gets submitted is whichever epoch
# actually looked best on held-out data.
best_val_loss = float("inf")
for epoch in range(EPOCHS):
    model.train()
    train_l1_loss = 0.0
    train_perceptual_loss = 0.0
    for y, cbcr in train_loader:
        y, cbcr = y.to(device), cbcr.to(device)
        opt.zero_grad()
        pred = model(y)

        # Calculate L1 loss
        l1_loss = l1_loss_fn(pred, cbcr)

        # Calculate Perceptual loss
        # y (luminance) is needed to convert CbCr back to RGB for VGG
        perceptual_loss = perceptual_loss_fn(y, pred, cbcr) # Pass y, predicted CbCr, and ground truth CbCr

        # Combine losses
        total_loss = (L1_WEIGHT * l1_loss) + (PERCEPTUAL_WEIGHT * perceptual_loss)

        total_loss.backward()
        opt.step()
        train_l1_loss += l1_loss.item() * y.size(0)
        train_perceptual_loss += perceptual_loss.item() * y.size(0)

    train_l1_loss /= len(train_ds)
    train_perceptual_loss /= len(train_ds)

    model.eval()
    val_l1_loss = 0.0
    val_perceptual_loss = 0.0
    with torch.no_grad():
        for y, cbcr in val_loader:
            y, cbcr = y.to(device), cbcr.to(device)
            pred = model(y)
            val_l1_loss += l1_loss_fn(pred, cbcr).item() * y.size(0)
            val_perceptual_loss += perceptual_loss_fn(y, pred, cbcr).item() * y.size(0)

    val_l1_loss /= len(val_ds)
    val_perceptual_loss /= len(val_ds)
    total_val_loss = (L1_WEIGHT * val_l1_loss) + (PERCEPTUAL_WEIGHT * val_perceptual_loss)

    scheduler.step()

    print("epoch %2d | lr %.2e | train L1 %.4f | train Perceptual %.4f | val L1 %.4f | val Perceptual %.4f | Total Val Loss %.4f" % (
        epoch, opt.param_groups[0]["lr"], train_l1_loss, train_perceptual_loss, val_l1_loss, val_perceptual_loss, total_val_loss))

    if total_val_loss < best_val_loss:
        best_val_loss = total_val_loss
        torch.save(model.state_dict(), "weights.pth")
        print("  (new best -- checkpoint saved to weights.pth)")

print("training done. best total val loss: %.4f" % best_val_loss)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


device: cuda
epoch  0 | lr 1.00e-03 | train L1 0.0637 | train Perceptual 1.4287 | val L1 0.0594 | val Perceptual 1.4451 | Total Val Loss 1.4748
  (new best -- checkpoint saved to weights.pth)
epoch  1 | lr 9.99e-04 | train L1 0.0616 | train Perceptual 1.4179 | val L1 0.0604 | val Perceptual 1.4611 | Total Val Loss 1.4912
epoch  2 | lr 9.98e-04 | train L1 0.0611 | train Perceptual 1.4085 | val L1 0.0581 | val Perceptual 1.3912 | Total Val Loss 1.4203
  (new best -- checkpoint saved to weights.pth)
epoch  3 | lr 9.96e-04 | train L1 0.0606 | train Perceptual 1.4030 | val L1 0.0625 | val Perceptual 1.4380 | Total Val Loss 1.4692
epoch  4 | lr 9.94e-04 | train L1 0.0602 | train Perceptual 1.3933 | val L1 0.0572 | val Perceptual 1.4227 | Total Val Loss 1.4513
epoch  5 | lr 9.91e-04 | train L1 0.0597 | train Perceptual 1.3888 | val L1 0.0572 | val Perceptual 1.4227 | Total Val Loss 1.4513
epoch  6 | lr 9.88e-04 | train L1 0.0592 | train Perceptual 1.3774 | val L1 0.0569 | val Perceptual 1.418

In [ ]:
# Self-check against the same metric the grader uses (PSNR), via the
# actual submission interface (load_model + colorize) -- not a
# shortcut through the training-time model object.
def psnr(pred, target, max_val=1.0):
    mse = float(np.mean((pred - target) ** 2))
    if mse == 0:
        return float("inf")
    return 10.0 * np.log10((max_val ** 2) / mse)


loaded = load_model("weights.pth")
psnrs = []
for y, cbcr in val_ds:
    y_np = y.squeeze(0).numpy()
    ground_truth_rgb = ycbcr_to_rgb(y_np, cbcr[0].numpy(), cbcr[1].numpy())
    pred_rgb = colorize(y_np, loaded)
    psnrs.append(psnr(pred_rgb, ground_truth_rgb))
print("mean val PSNR: %.2f dB over %d held-out images" % (float(np.mean(psnrs)), len(psnrs)))

In [ ]:
# =======================================================================
# DEMO -- file loading happens here, OUTSIDE colorize() (fixed rule 3)
# =======================================================================
import matplotlib.pyplot as plt

model = load_model("weights.pth")

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row, idx in enumerate([0, 1, 2]):
    y_sample, cbcr_sample = val_ds[idx]
    y_np = y_sample.squeeze(0).numpy()
    ground_truth_rgb = ycbcr_to_rgb(y_np, cbcr_sample[0].numpy(), cbcr_sample[1].numpy())
    pred = colorize(y_np, model)

    axes[row, 0].imshow(y_np, cmap="gray")
    axes[row, 1].imshow(pred)
    axes[row, 2].imshow(ground_truth_rgb)
    if row == 0:
        for ax, title in zip(axes[row], ["input (gray)", "predicted color", "ground truth"]):
            ax.set_title(title)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()